# 【notebook｜N-19】Python 面向对象与协议
> 对应理论篇：[`PY2-Python语言II.md`](../../00-预备/PY2-Python语言II.md)（doccode = `PY2`）
> 对应代码：`00-预备/code/py2_oop_protocols.py`（Tag：【源代码｜F-pre-py2】）
> 对接本仓：`01-基础/code/from_scratch/data.py`（`F-b01-data`）、`model.py`（`F-b01-model`）
> **Colab**：CPU 运行时，纯标准库，全部运行约 5 秒。

## 学习目标
1. 搞清类属性 vs 实例属性、继承与 MRO、`super()` 的语义；
2. 掌握数据模型（`__repr__`/`__len__`/`__getitem__`/`__eq__`/`__hash__`/`__enter__`/`__exit__`）；
3. 会用 `property`、`dataclass`、`typing`、`enum`；
4. 能读懂异常链、上下文管理器、装饰器与生成器，并看懂 `data.py`。


## 1. 类与实例：类属性、实例属性与运算符

对照 C++：`self` 是显式参数，方法都是“虚函数”；类属性近似 `static` 成员。


In [ ]:
# 【N-19.c1】
from dataclasses import dataclass, field
from enum import Enum
from typing import Sequence, Protocol, runtime_checkable


class Vec2:
    dim = 2                      # 类属性：所有实例共享
    __slots__ = ("x", "y")       # 去掉 __dict__，属性只能落槽位

    def __init__(self, x, y):
        self.x, self.y = x, y    # 实例属性：每个实例各自一份

    def __repr__(self):
        return f"Vec2({self.x!r}, {self.y!r})"

    def __eq__(self, other):
        if not isinstance(other, Vec2):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))

    def __add__(self, other):
        return Vec2(self.x + other.x, self.y + other.y)


a, b = Vec2(1, 2), Vec2(3, 4)
print(a, "+", b, "=", a + b)
print("a == Vec2(1,2):", a == Vec2(1, 2), "| hash 相等:", hash(a) == hash(Vec2(1, 2)))
print("类属性 Vec2.dim =", Vec2.dim)


## 2. 数据模型：只靠特殊方法获得能力

容器协议（`__len__`/`__getitem__`/`__iter__`）是结构化接口；`__eq__` 与 `__hash__` 必须满足“相等则同哈希”的契约（见【命题 PY2.2.3】、【命题 PY2.2.4】）。


In [ ]:
# 【N-19.c2】
class Buffer:
    def __init__(self, data):
        self._data = list(data)

    def __len__(self):
        return len(self._data)

    def __getitem__(self, i):
        return self._data[i]

    def __iter__(self):
        return iter(self._data)

    def __repr__(self):
        return f"Buffer({self._data!r})"


buf = Buffer([10, 20, 30])
print("len =", len(buf), "| buf[1] =", buf[1], "| list =", list(buf))
print("20 in buf:", 20 in buf)
print("repr（开发者视角）:", repr(buf))

# __eq__ 默认按身份；自定义 __eq__ 后 __hash__ 被置空（不可哈希）
class NoHash:
    def __eq__(self, other):
        return isinstance(other, NoHash)

print("NoHash().__hash__ is None:", NoHash.__hash__ is None)


## 3. 属性：property 与 __slots__ 内存

`property` 让方法以属性语法读写，并在 setter 中校验；`__slots__` 去掉实例字典以省内存（见【定义 PY2.3.1】、【定义 PY2.3.3】）。


In [ ]:
# 【N-19.c3】
class Counter:
    def __init__(self, value=0):
        self._value = value

    @property
    def value(self):
        return self._value

    @value.setter
    def value(self, new):
        if new < 0:
            raise ValueError("value 必须非负")
        self._value = new

    @property
    def doubled(self):
        return self._value * 2


c = Counter(5)
c.value = 7
print("value =", c.value, "| doubled =", c.doubled)
try:
    c.value = -1
except ValueError as exc:
    print("setter 校验:", exc)

import sys

class WithDict:
    def __init__(self):
        self.a, self.b = 0, 0

class WithSlots:
    __slots__ = ("a", "b")
    def __init__(self):
        self.a, self.b = 0, 0

d, s = WithDict(), WithSlots()
print("普通实例内存:", sys.getsizeof(d) + sys.getsizeof(d.__dict__), "| slots 实例:", sys.getsizeof(s))


## 4. dataclass / typing / enum

`@dataclass` 自动生成 `__init__`/`__repr__`/`__eq__`；`Protocol` 表达结构化子类型；`Enum` 表达具名常量（见 §4）。


In [ ]:
# 【N-19.c4】
@dataclass
class Batch:
    x: Sequence[int]
    y: Sequence[int]
    meta: dict = field(default_factory=dict)   # 可变默认值必须用 default_factory


class Color(Enum):
    RED = 1
    GREEN = 2
    BLUE = 3


@runtime_checkable
class TokenizerLike(Protocol):
    def encode(self, text: str, add_eos: bool = False) -> list[int]:
        ...

    def decode(self, ids: Sequence[int], skip_special: bool = True) -> str:
        ...


class SimpleTokenizer:
    def __init__(self, chars):
        self.itos = {i: ch for i, ch in enumerate(sorted(set(chars)))}
        self.stoi = {ch: i for i, ch in self.itos.items()}

    def encode(self, text, add_eos=False):
        return [self.stoi[ch] for ch in text if ch in self.stoi]

    def decode(self, ids, skip_special=True):
        return "".join(self.itos[int(i)] for i in ids)


print("Batch:", Batch(x=[1, 2], y=[2, 3]))
print("Enum:", Color.RED, Color.RED.value, "| is 单例:", Color.RED is Color.RED)
tok = SimpleTokenizer("abc")
print("Protocol isinstance（不继承也满足）:", isinstance(tok, TokenizerLike))
print("decode:", tok.decode(tok.encode("cab")))


## 5. 异常模型：try/except/else/finally 与异常链

对照 C：异常把错误通道与正常返回通道分离；`raise ... from` 保留根因（见【定义 PY2.5.3】）。


In [ ]:
# 【N-19.c5】
class TokenizerError(Exception):
    """本模块分词相关错误的基类。"""


def parse_ids(text):
    try:
        ids = [int(p) for p in text.split(",")]
    except ValueError as exc:
        raise TokenizerError(f"非法 token 序列: {text!r}") from exc
    else:
        return ids
    finally:
        pass  # 可在此做清理


print("解析成功:", parse_ids("1,2,3"))
try:
    parse_ids("1,x,3")
except TokenizerError as exc:
    print("捕获:", type(exc).__name__, "| 根因:", type(exc.__cause__).__name__)


## 6. 上下文管理器与 `with`

对照 RAII：`with` 在进入/退出时调用 `__enter__`/`__exit__`，异常也可交给 `__exit__` 处理（见 §6）。


In [ ]:
# 【N-19.c6】
import os
import time
from contextlib import contextmanager


class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        self.elapsed = 0.0
        return self

    def __exit__(self, exc_type, exc, tb):
        self.elapsed = time.perf_counter() - self.start
        return False    # 不吞异常


@contextmanager
def pushed(stack, item):
    stack.append(item)
    try:
        yield item
    finally:
        stack.pop()


with Timer() as t:
    sum(range(100000))
print(f"Timer: {t.elapsed:.3e}s")

stack = []
with pushed(stack, "token") as item:
    print("with 内:", item, stack)
print("with 后:", stack)


def atomic_write(path, text):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(text)
    os.replace(tmp, path)   # 同文件系统内原子替换


p = "/tmp/py2_n19_demo.txt"
atomic_write(p, "hello")
print("atomic_write:", open(p, encoding="utf-8").read(), "| tmp 存在:", os.path.exists(p + ".tmp"))


## 7. 装饰器与高阶函数

`@deco` 等价 `f = deco(f)`；带参装饰器需三层；`functools.wraps` 保元数据（见 §7）。


In [ ]:
# 【N-19.c7】
import functools


def retry(times):
    def deco(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            last = None
            for _ in range(times):
                try:
                    return fn(*args, **kwargs)
                except RuntimeError as exc:
                    last = exc
            raise last
        return wrapper
    return deco


calls = {"n": 0}


@retry(3)
def flaky():
    calls["n"] += 1
    if calls["n"] < 3:
        raise RuntimeError("还没成功")
    return "success"


print("retry 结果:", flaky(), "| 调用次数:", calls["n"], "| __name__:", flaky.__name__)


@functools.lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)


print("fib(20):", fib(20), "| cache:", fib.cache_info().currsize)
print("sorted(key=abs):", sorted([-3, 1, 2, -5], key=abs))


## 8. 迭代器与生成器

生成器惰性求值、内存 O(1)；生成器表达式与列表推导式的取舍见【命题 PY2.8.3】。


In [ ]:
# 【N-19.c8】
import itertools


def windows(it, n):
    buf = []
    for x in it:
        buf.append(x)
        if len(buf) == n:
            yield tuple(buf)
            buf.pop(0)


print("windows:", list(windows(range(5), 3)))
print("islice(count):", list(itertools.islice(itertools.count(1), 5)))
print("chain:", list(itertools.chain([1, 2], [3, 4])))

# 生成器表达式不物化中间列表
print("pipeline sum:", sum(v * v for v in [-2, -1, 0, 1, 2] if v >= 0))


## 思考题
1. 为什么 `self.x = x` 写的是实例字典，而 `C.x = 1` 改的是类字典？两者的查找顺序如何影响结果？
2. 自定义 `__eq__` 后为什么 `__hash__` 变为 `None`？这对把对象放进 `set` 有何影响？
3. 生成器表达式与列表推导式在“只消费一次”和“多次遍历”场景下如何取舍？
4. `with` 与 C++ RAII 在异常路径上的行为有何异同？

## 改动实验（任选 2–3 个）
1. 给 `Vec2` 增加 `__mul__`（数乘）与 `__radd__`（与数相加），观察 `2 + v` 的分派。
2. 用 `@dataclass(frozen=True)` 定义 `Point`，验证不可变与可哈希。
3. 用 `contextlib.suppress` 改写 `parse_ids` 的异常处理。

## 预期输出
```
Vec2(4, 6)
len = 3 | buf[1] = 20 | list = [10, 20, 30]
value = 7 | doubled = 14
setter 校验: value 必须非负
Protocol isinstance（不继承也满足）: True
捕获: TokenizerError | 根因: ValueError
with 后: []
retry 结果: success | 调用次数: 3
fib(20): 6765
windows: [(0, 1, 2), (1, 2, 3), (2, 3, 4)]
```
